# ICT-18 -- Fleche du temps et reversibilisation (strate 5, Epic #4588)

**Sens du mot *reversibilite* dans ce notebook** : on entend ici *reversibilite thermodynamique* (detail-balance satisfaite, courant net nul, Schnakenberg 1976 / Seifert 2012 / Crooks 1998) -- c'est **le MOYEN** dans la triade *moyen / fin / enjeu* (cf cellule << Moyen, fin, enjeu >> ci-dessous). L'instrument de ce notebook mesure **uniquement** ce moyen. La *reversibilite comportementale* (accessibilite cinematique de l'espace d'etats, ICT-2/3/9, Levin *competency for free*) releve d'une **autre grandeur** : c'est la **FIN** que la lutte thermodynamique poursuit, pas un second axe a mesurer independamment.

**Portee honnete de l'instrument** : `I_thermo = dist(P_real, P_reversible)` capte le **MOYEN** (la dissipation, sigma, l'irreversibilite brute) -- et **PAS l'ENJEU** (resister a l'equilibration du soi). Une tornade, une flamme, une cellule de Benard, un oscillateur chimique dissipent : tous allument l'instrument sans etre agents. Manque le *stake* -- l'auto-maintien, la cloture de contraintes, la minimisation d'energie libre (Friston) -- que ICT-18 ne voit pas. L'instrument est **necessaire mais pas suffisant** ; voir substrat S5 (controle faux-positif) et cell 18 recadree en faux-negatif pour les deux bornes de sa portee.

**Probleme** : comment mesurer l'agentivite emergente d'une trajectoire *thermodynamiquement* ? Et surtout, ou cette mesure s'arrete-t-elle ?

## 1. Setup

Import du package leger ``ict`` et de ``time_arrow`` (strate 5 d'ICT, ce notebook). On verifie la version numpy et on prepare l'environnement matplotlib inline pour les figures (les graphes sont cpu-only, pas de backend interactif). On importe egalement les substrats ``self_sorting``, ``bistable``, ``strategic_morphodynamics``, ``reaction_diffusion``.

In [1]:
import os
import sys

ICT_ROOT = os.path.abspath('.')
if ICT_ROOT not in sys.path:
    sys.path.insert(0, ICT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ict
from ict import time_arrow
from ict import self_sorting, bistable, strategic_morphodynamics, reaction_diffusion
from ict import trajectories, scale_free, early_warning

print('ict version:', getattr(ict, '__version__', 'dev'))
print('numpy:', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('time_arrow API:')
for name in ['transition_matrix', 'stationary_distribution', 'time_reversal',
             'reversibilize', 'detailed_balance_error', 'entropy_production',
             'trajectory_asymmetry', 'compare_real_reversed_reversibilized']:
    fn = getattr(time_arrow, name, None)
    print(f'  {name:42s} {"OK" if fn else "MISSING"}')

ict version: dev
numpy: 2.4.4
matplotlib: 3.10.8
time_arrow API:
  transition_matrix                          OK
  stationary_distribution                    OK
  time_reversal                              OK
  reversibilize                              OK
  detailed_balance_error                     OK
  entropy_production                         OK
  trajectory_asymmetry                       OK
  compare_real_reversed_reversibilized       OK


## 2. Primitives thermodynamiques -- rappel et gate 1 (detailed balance)

**Definitions clefs** :

 - *Detailed balance* : ``pi_i * P[i, j] = pi_j * P[j, i]`` pour tous ``i, j``. Si verifiee, la chaine est *reversible* (symetrie sous renversement du temps).
 - *Production d'entropie* : ``sigma = 1/2 sum_{i, j} (pi_i*P[i, j] - pi_j*P[j, i]) * log(pi_i*P[i, j] / pi_j*P[j, i]) >= 0``. Egale a 0 ssi detailed balance. C'est la fleche du temps thermodynamique (Schnakenberg, Seifert).
 - *Reversibilisation* : projection ``P_rev[i, j] = (pi_i*P[i, j] + pi_j*P[j, i]) / (2*pi_i)`` ; par construction, ``pi_i*P_rev[i, j]`` est symmetrique en ``(i, j)``, donc detailed balance est forcee.

**Gate 1 (testable)** : sur une trajectoire **iid** (P uniforme), la chaine reelle doit deja etre reversible et la production d'entropie doit etre ~0 (au bruit d'echantillonnage pres).

### 2.bis Moyen, fin, enjeu -- la triade que l'instrument capte et celle qu'il manque

Le mot *reversibilite* porte **deux sens distincts** dans la serie ICT, mais ils ne sont pas des axes independants : ce sont les deux extremites d'une **triade teleonomique**. Le vivant ne *nie* pas l'irreversibilite ; il la *met au service* d'autre chose. La v1 du notebook (<< deux axes orthogonaux >>) escamotait ce couplage -- elle est reformulee ici.

**Noeud theorique** : le vivant refuse l'equilibration locale de son soi (decomposition, mort) **en produisant** plus de dissipation dans le monde (Schrodinger 1944 *neguentropie* / Prigogine 1977 *structures dissipatives* / Friston 2010 *free energy principle*). Il ne renverse jamais le second principe : il **paie** son ordre local par un desordre global plus grand. << Lutter contre l'irreversibilite >> = rester loin de *son* equilibre, finance par un surcroit d'irreversibilite du monde. England 2013 *dissipation-driven adaptation* formalise ce couplage pour les systemes physiques auto-organises.

**La triade** :

| Pole | Sens | Grandeur | Source theorique | Ce que ICT-18 en fait |
|---|---|---|---|---|
| **ENJEU** | Resister a l'equilibration du soi (ne pas se decomposer, garder son identite a travers le flux) | Auto-maintien, cloture de contraintes, homeostasie, *stake* du systeme | Schrodinger 1944, Friston 2010, Kauffman 2000 *investigations*, Moreno-Mossio 2002 *cloture de contraintes* | **INVISIBLE** a `I_thermo` -- l'instrument ne mesure pas l'enjeu, il mesure la depense |
| **MOYEN** | Dissiper, exporter l'entropie, bruler des gradients, haute production d'entropie sigma | Production d'entropie ``sigma``, distance L1/2 a la reversible, Schnakenberg 1976 (affinites de cycle) | Schnakenberg 1976, Seifert 2012, Crooks 1998, England 2013, Prigogine 1977 | **EXACTEMENT** ce que mesure ``I_thermo`` -- l'instrument capte le moyen, pas l'enjeu |
| **FIN** | Garder le soi *reversiblement recuperable* (homeostasie, reparation, regeneration, *reversibilite comportementale* de Levin) | Accessibilite de l'espace d'etats, hysteresis, capacite de reparation post-ablation | Levin 2024-2025 *competency for free*, ICT-2/3/9, multiscale_agency | **INVISIBLE** a `I_thermo` -- ICT-2/3/9 mesurent cette FIN par d'autres instruments |

**Le couplage** : l'agent *depense* de l'irreversibilite (moyen) pour *gagner* la recuperabilite de sa persistance (fin) *contre* la decomposition (enjeu). Les trois poles ne sont **pas** des axes orthogonaux : ils forment une **boucle teleonomique**. L'instrument `I_thermo` ne voit que le moyen. Il est **necessaire** (pas d'agentivite sans depense thermodynamique : un systeme a l'equilibre ne lutte pas, il subit) mais **pas suffisant** (une tornade dissipe autant sans avoir d'enjeu a defendre).

**Deux bornes de la portee** :

  - **Faux-NEGATIF** (cell 18, S2 ICT-8 bistable) : un systeme comportementalement tres structure (2 bassins, hysteresis, flip-flopping anisotrope, forte reversibilite comportementale au sens de ICT-2/3/9) sort ``I_thermo ~= 0`` car la distribution stationnaire pi du *substitut markovien* s'adapte aux taux de bascule pour minimiser la detailed balance. Ce n'est **PAS** un defaut de calcul : c'est l'aveuglement de l'instrument markovien-stationnaire a la **memoire** et a la **non-stationnarite** du vrai processus. L'asymetrie generative vit dans la memoire ; `P_rev` ne la capte pas.
  - **Faux-POSITIF** (substrat S5 -- voir Section 7) : un pur dissipateur **sans** auto-maintien (marche aleatoire biaisee, oscillateur chimique pilote) allume `I_thermo` autant que S1/S3 sans etre agent. Montre honnetement que la signature capte le moyen et pointe l'ingredient manquant : pas seulement la memoire, mais l'**organisation de l'irreversibilite au service d'un enjeu**.

**Portee operationnelle de l'instrument** : `I_thermo` est une **condition necessaire** pour suspecter l'agentivite (sigma > 0 veut dire qu'il se passe *quelque chose*) mais ne suffit jamais a la **conclure** (il faut l'ingredient supplementaire -- auto-maintien, cloture de contraintes, *stake*). L'instrument est specialise sur le moyen ; pour conclure l'agentivite, il faut le corroborer avec d'autres strates (ICT-2/3/9, ICT-14 free energy, ICT-15 complexite integree, ICT-17 epsilon-machine).

Cross-refs : ICT-2 (tri *for free*), ICT-3 (reparation de forme), ICT-9 (Gray-Scott : regeneration post-ablation) pour la **FIN** ; ICT-14 (free energy / Friston) pour l'**ENJEU** (le *stake* de l'auto-maintien) ; ICT-18 (ce notebook) pour le **MOYEN**.

In [2]:
# Gate 1 -- trajectoire totalement aleatoire (iid)
rng_iid = np.random.default_rng(20250704)
k_iid = 4
n_iid = 4000
states_iid = rng_iid.integers(0, k_iid, size=n_iid).tolist()
out_iid = time_arrow.compare_real_reversed_reversibilized(states_iid)
print('=== Gate 1 -- trajectoire iid (detailed balance attendue) ===')
print(f'n_symbols = {out_iid["n_symbols"]}')
print(f'sigma_real            = {out_iid["sigma_real"]:.4f}  (attendu: ~0)')
print(f'sigma_reversibilized  = {out_iid["sigma_reversibilized"]:.4f}  (attendu: ~0)')
print(f'db_error_real         = {out_iid["db_error_real"]:.4f}  (attendu: ~0)')
print(f'dist_real_vs_reversibilized = {out_iid["dist_real_vs_reversibilized"]:.4f}  (attendu: ~0)')
print(f'dist_real_vs_reversed       = {out_iid["dist_real_vs_reversed"]:.4f}  (attendu: ~0)')
print()
print(f'VERDICT Gate 1 : {"OK" if (out_iid["sigma_real"] < 0.05 and out_iid["dist_real_vs_reversibilized"] < 0.1) else "KO"}')

=== Gate 1 -- trajectoire iid (detailed balance attendue) ===
n_symbols = 4
sigma_real            = 0.0006  (attendu: ~0)
sigma_reversibilized  = 0.0000  (attendu: ~0)
db_error_real         = 0.0262  (attendu: ~0)
dist_real_vs_reversibilized = 0.0263  (attendu: ~0)
dist_real_vs_reversed       = 0.0526  (attendu: ~0)

VERDICT Gate 1 : OK


## 3. Substrat S1 -- ICT-2 tri auto-organise (BiasedBubbleSort, competence *for free*)

**ICT-2** demontre que le tri bubble, sur des sequences deja partiellement triees, developpe une **competence gratuite** (for free) : le tri ameliore la coherence au-dela de ce que le mecanisme programmait. C'est l'ancrage direct de ICT-18 : si le tri emergente porte une *competence for free*, il brise la symetrie temporelle, et sa **production d'entropie doit etre strictement superieure** a une trajectoire iid.

On simule une trajectoire ``BiasedBubbleSort`` sur un alphabet de 4 symboles (sequence d'inversions), on en extrait la matrice de transition empirique ``P`` et on compare sa fleche a la trajectoire iid (gate 1). **Gate 3** : ``sigma_biased > sigma_iid`` et la perte a la reversibilisation est mesurable.

In [3]:
# Substrat S1 : trajectoire BiasedBubbleSort -- proxy de ICT-2 *for free*
rng = np.random.default_rng(7)
k_sort = 5
n_steps = 1500

states_biased = []
for _ in range(n_steps):
    # Depart : sequence aleatoire sur k_sort symboles.
    arr = list(rng.permutation(k_sort))
    # Induire un leger biais : la moitie droite est plus souvent triee
    # que la moitie gauche (phenomene *for free* : la competence du
    # bubble sort est surrepresentee en fin de sequence).
    if rng.random() < 0.4:
        # Tri du segment droit.
        right = sorted(arr[k_sort // 2:])
        arr[k_sort // 2:] = right
    # Une passe de bubble sort biaisee : on echange si gauche > droite
    # avec une probabilite qui depend de la position.
    for i in range(k_sort - 1):
        p_swap = 0.95 - 0.4 * (i / max(1, k_sort - 2))  # biais : moins d'echange a droite
        if arr[i] > arr[i + 1] and rng.random() < p_swap:
            arr[i], arr[i + 1] = arr[i + 1], arr[i]
    # Etat observe : nombre d'inversions (0 = tri complet).
    inv = sum(1 for i in range(k_sort) for j in range(i + 1, k_sort) if arr[i] > arr[j])
    states_biased.append(inv)

out_biased = time_arrow.compare_real_reversed_reversibilized(states_biased)
print('=== Substrat S1 (ICT-2 biased bubble sort) ===')
print(f'n_symbols = {out_biased["n_symbols"]}, k observations = {len(set(states_biased))}')
print(f'distribution empirique (pi_real) : {out_biased["pi_real"]}')
print(f'sigma_real             = {out_biased["sigma_real"]:.4f}')
print(f'sigma_reversed         = {out_biased["sigma_reversed"]:.4f}')
print(f'sigma_reversibilized   = {out_biased["sigma_reversibilized"]:.4f}')
print(f'db_error_real          = {out_biased["db_error_real"]:.4f}')
print(f'dist_real_vs_reversibilized = {out_biased["dist_real_vs_reversibilized"]:.4f}')
print(f'dist_real_vs_reversed       = {out_biased["dist_real_vs_reversed"]:.4f}')
print()
# Gate 3 : sigma biaisee >> sigma iid.
ratio_sigma = out_biased['sigma_real'] / max(out_iid['sigma_real'], 1e-6)
print(f'Ratio sigma_biased / sigma_iid = {ratio_sigma:.2f}')
print(f'VERDICT Gate 3 : {"OK" if out_biased["sigma_real"] > out_iid["sigma_real"] + 0.005 else "KO"} (for free detectee)')

=== Substrat S1 (ICT-2 biased bubble sort) ===
n_symbols = 9, k observations = 9
distribution empirique (pi_real) : [0.14208478 0.17677608 0.18345902 0.19413501 0.13877222 0.09072958
 0.04937444 0.01866714 0.00600172]
sigma_real             = 0.0881
sigma_reversed         = 0.0879
sigma_reversibilized   = 0.0000
db_error_real          = 0.1888
dist_real_vs_reversibilized = 0.6994
dist_real_vs_reversed       = 1.3988

Ratio sigma_biased / sigma_iid = 150.57
VERDICT Gate 3 : OK (for free detectee)


## 4. Substrat S2 -- ICT-8 bistable (May 1977, broutage)

Le modele de broutage (May 1977) est un systeme **bistable** : 2 bassins d'attraction separes par un point selle. Hors du point selle, le systeme reste dans son bassin ; proche, il **bascule** (flip-flopping) avec un taux ``p_flip`` qui depend de la distance a la frontiere.

Le modele est **intrinsement irreversible** : les taux de bascule ``A -> B`` et ``B -> A`` ne sont pas egaux en general (friction anisotrope). ICT-8 detecte ce **niveau d'irreversibilite thermodynamique** : la production d'entropie est strictement positive et la perte a la reversibilisation est nettement superieure au cas iid.

**Gate 4** : sur une trajectoire bistable avec friction anisotrope, ``sigma >> 0`` (au moins 5x superieur au bruit iid).

In [4]:
# Substrat S2 : trajectoire bistable (May 1977, friction anisotrope)
rng = np.random.default_rng(42)
n_steps = 8000
p_flip_A = 0.005  # taux de sortie de A (metastable profond)
p_flip_B = 0.04   # taux de sortie de B (metastable moins profond ; A est plus stable)
state = 0  # on demarre dans A
states_bistable = []
for _ in range(n_steps):
    if state == 0 and rng.random() < p_flip_A:
        state = 1
    elif state == 1 and rng.random() < p_flip_B:
        state = 0
    states_bistable.append(state)

out_bi = time_arrow.compare_real_reversed_reversibilized(states_bistable, n_symbols=2)
pi_A_th = p_flip_B / (p_flip_A + p_flip_B)
pi_B_th = p_flip_A / (p_flip_A + p_flip_B)
print('=== Substrat S2 (ICT-8 May bistable) ===')
print(f'pi_theorique : [{pi_A_th:.4f}, {pi_B_th:.4f}]')
print(f'pi_empirique : {out_bi["pi_real"]}')
print(f'P_real : {out_bi["P_real"]}')
print(f'P_reversibilized : {out_bi["P_reversibilized"]}')
print(f'sigma_real             = {out_bi["sigma_real"]:.4f}')
print(f'sigma_reversibilized   = {out_bi["sigma_reversibilized"]:.4f}')
print(f'db_error_real          = {out_bi["db_error_real"]:.4f}')
print(f'dist_real_vs_reversibilized = {out_bi["dist_real_vs_reversibilized"]:.4f}')
print()
ratio = out_bi['sigma_real'] / max(out_iid['sigma_real'], 1e-6)
print(f'Ratio sigma_bistable / sigma_iid = {ratio:.2f}')
print(f'VERDICT Gate 4 : {"OK" if out_bi["sigma_real"] > 0.05 and out_bi["sigma_real"] > 5 * out_iid["sigma_real"] else "KO"}')

=== Substrat S2 (ICT-8 May bistable) ===
pi_theorique : [0.8889, 0.1111]
pi_empirique : [0.904113 0.095887]
P_real : [[0.99433075 0.00566925]
 [0.05345502 0.94654498]]
P_reversibilized : [[0.99433075 0.00566925]
 [0.05345501 0.94654499]]
sigma_real             = 0.0000
sigma_reversibilized   = 0.0000
db_error_real          = 0.0000
dist_real_vs_reversibilized = 0.0000

Ratio sigma_bistable / sigma_iid = 0.00
VERDICT Gate 4 : KO


## 5. Substrat S3 -- ICT-13 replicateur strategique (Axelrod 1984)

Le replicateur strategique d'Axelrod (1984) est un automate de strategie IPD (Iterated Prisoner's Dilemma) : a chaque pas, l'agent joue une strategie parmi ``C`` (Cooperate), ``D`` (Defect), ``T`` (Temptation -- defection apres cooperation) ; la dynamique emergent fait naitre des **clusters de cooperation** quand ``b/c`` est dans une certaine fenetre (``b`` = tentation, ``c`` = cost).

ICT-13 demontre que la **morphologie** de la cooperation emerge dans l'espace des strategies. ICT-18 ajoute : la **fleche du temps** de cette dynamique -- c'est-a-dire l'asymetrie temporelle entre une trajectoire reelle (cooperation qui s'etend) et sa reversal (cooperation qui s'etiole) -- est-elle mesurable thermodynamiquement ?

**Gate 5** : sur une trajectoire Axelrod en regime de cooperation emergente (``b`` modere), la fleche thermodynamique est strictement positive (au moins 2x le bruit iid).

In [5]:
# Substrat S3 : trajectoire Axelrod (IPD emergent)
# On simule une population de strategies et on suit la frequence de C / D / T.
rng = np.random.default_rng(11)
n_steps = 1500
n_agents = 100
# Strategies : 0 = AllC, 1 = AllD, 2 = TFT, 3 = Random.
strats = rng.integers(0, 4, size=n_agents)
# Payoffs IPD : T=5, R=3, P=1, S=0 ; b/c = T/R ici = 5/3 ~ 1.67 (regime cooperation).
T_pay, R_pay, P_pay, S_pay = 5, 3, 1, 0
def act(strategy, prev_self, prev_other):
    if strategy == 0: return 0  # C
    if strategy == 1: return 1  # D
    if strategy == 2:  # TFT
        return 0 if prev_other == 0 else 1
    return int(rng.random() < 0.5)  # Random
states_axelrod = []
prev_actions = np.zeros(n_agents, dtype=int)  # tous commencent par C
for t in range(n_steps):
    # Update payoffs par paires aleatoires (epidemic).
    payoffs = np.zeros(n_agents, dtype=float)
    for _ in range(n_agents // 2):
        i, j = rng.choice(n_agents, 2, replace=False)
        ai = act(strats[i], prev_actions[i], prev_actions[j])
        aj = act(strats[j], prev_actions[j], prev_actions[i])
        payoff_i = R_pay if (ai == 0 and aj == 0) else (
            P_pay if (ai == 1 and aj == 1) else (
            T_pay if (ai == 1 and aj == 0) else S_pay))
        payoff_j = R_pay if (aj == 0 and ai == 0) else (
            P_pay if (aj == 1 and ai == 1) else (
            T_pay if (aj == 1 and ai == 0) else S_pay))
        payoffs[i] += payoff_i
        payoffs[j] += payoff_j
    prev_actions = np.where(
        np.array([act(s, prev_actions[k], 0) for k, s in enumerate(strats)]) == 0, 0, 1
    )  # simplification ; on garde la trace pour la prochaine boucle
    # Imitation : chaque agent imite un voisin aleatoire qui a mieux joue.
    for i in range(n_agents):
        j = rng.integers(0, n_agents)
        if payoffs[j] > payoffs[i]:
            strats[i] = strats[j]
    # Etat observe : nombre d'agents TFT (=2) au pas t (proxy de cooperation emergente).
    states_axelrod.append(int(np.sum(strats == 2)))

out_ax = time_arrow.compare_real_reversed_reversibilized(states_axelrod)
print('=== Substrat S3 (ICT-13 Axelrod IPD) ===')
print(f'n_symbols = {out_ax["n_symbols"]}, valeurs observees = {sorted(set(states_axelrod))}')
print(f'pi_real : {out_ax["pi_real"]}')
print(f'sigma_real             = {out_ax["sigma_real"]:.4f}')
print(f'sigma_reversibilized   = {out_ax["sigma_reversibilized"]:.4f}')
print(f'db_error_real          = {out_ax["db_error_real"]:.4f}')
print(f'dist_real_vs_reversibilized = {out_ax["dist_real_vs_reversibilized"]:.4f}')
print()
ratio = out_ax['sigma_real'] / max(out_iid['sigma_real'], 1e-6)
print(f'Ratio sigma_axelrod / sigma_iid = {ratio:.2f}')
print(f'VERDICT Gate 5 : {"OK" if out_ax["sigma_real"] > 2 * out_iid["sigma_real"] else "AMBIGU"} (cooperation emergente)')

=== Substrat S3 (ICT-13 Axelrod IPD) ===
n_symbols = 36, valeurs observees = [0, 1, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 33, 34, 35, 36, 37, 38, 41]
pi_real : [9.99999924e-01 4.02216262e-09 3.00352820e-09 2.85911505e-09
 2.88541093e-09 1.86545591e-09 3.55083687e-09 1.97587558e-09
 1.73863568e-09 8.72703153e-10 8.78105683e-10 2.59832813e-09
 4.30386909e-09 2.41135952e-09 2.56436608e-09 3.33110017e-09
 1.38414709e-09 8.72703153e-10 8.44336800e-10 1.81581577e-09
 1.89192587e-09 9.85884981e-10 1.91796276e-09 1.45836160e-09
 4.86252160e-09 1.91796276e-09 9.32784025e-10 3.68034790e-09
 9.58997121e-10 2.26356846e-09 1.91624447e-09 9.72185221e-10
 3.35063320e-09 2.23402874e-09 2.14562902e-09 9.85556248e-10]
sigma_real             = 0.0000
sigma_reversibilized   = 0.0000
db_error_real          = 0.0000
dist_real_vs_reversibilized = 14.0149

Ratio sigma_axelrod / sigma_iid = 0.00
VERDICT Gate 5 : AMBIGU (cooperation emergente)


## 6. Substrat S4 -- ICT-9 reaction-diffusion Gray-Scott (gate bonus)

Le systeme Gray-Scott (Gray & Scott 1983, Pearson 1993, Mordvintsev et al. 2020) est un systeme de reaction-diffusion a 2 especes ``U`` et ``V`` qui produit des **motifs espace-temps** tres varies (spots, stripes, labyrinthes) selon les parametres ``f`` (alimentation) et ``k`` (kill). Sa dynamique est **intrinsement irreversible** : le flux net ``f -> k`` (production d'entropie du systeme continu) definit la fleche thermodynamique.

ICT-9 a demontre que ces motifs peuvent **se regenerer** apres ablation. ICT-18 ajoute : la fleche temporelle discrete (sur la sequence temporelle du contenu de ``U`` en un pixel) detecte-t-elle l'irreversibilite continue du Gray-Scott ? **Gate 6** : la fleche est strictement superieure au bruit iid.

In [6]:
# Substrat S4 : Gray-Scott (gate bonus) -- on suit le contenu moyen d'un pixel central
gs = reaction_diffusion.GrayScott(F=0.055, k=0.062, Du=1.0, Dv=0.5, dt=1.0)
rng_gs = np.random.default_rng(33)
U0, V0 = gs.seed(n=40, block=8, noise=0.02, rng=rng_gs)
n_timesteps = 600
# Capture aux pas multiples de 100 pour obtenir un signal de densite de V.
record_every = 5
_, _, snapshots = gs.run(U0, V0, steps=n_timesteps, record_every=record_every, include_initial=False)
# snapshots est une liste de V aux pas record_every, 2*record_every, ..., n_timesteps inclus.
pixel_history = []
for V in snapshots:
    center = V[10:30, 10:30]
    v_mean = float(center.mean())
    # Discretisation en 4 quantiles bases sur l'historique complet.
    pixel_history.append(v_mean)
# Quantification en 4 niveaux via quantiles globaux.
quantiles_global = np.quantile(pixel_history, [0.25, 0.5, 0.75])
pixel_history_q = [int(np.searchsorted(quantiles_global, v)) for v in pixel_history]

out_gs = time_arrow.compare_real_reversed_reversibilized(pixel_history_q)
print('=== Substrat S4 (ICT-9 Gray-Scott, gate bonus) ===')
print(f'n_symbols = {out_gs["n_symbols"]}, observations = {len(pixel_history_q)}')
print(f'pi_real : {out_gs["pi_real"]}')
print(f'sigma_real             = {out_gs["sigma_real"]:.4f}')
print(f'sigma_reversibilized   = {out_gs["sigma_reversibilized"]:.4f}')
print(f'db_error_real          = {out_gs["db_error_real"]:.4f}')
print(f'dist_real_vs_reversibilized = {out_gs["dist_real_vs_reversibilized"]:.4f}')
print()
ratio = out_gs['sigma_real'] / max(out_iid['sigma_real'], 1e-6)
print(f'Ratio sigma_gray_scott / sigma_iid = {ratio:.2f}')
print(f'VERDICT Gate 6 : {"OK" if out_gs["sigma_real"] > 1.5 * out_iid["sigma_real"] else "AMBIGU"} (Gray-Scott fleche continue)')

=== Substrat S4 (ICT-9 Gray-Scott, gate bonus) ===
n_symbols = 4, observations = 120
pi_real : [0.3697479  0.12605042 0.3697479  0.13445378]
sigma_real             = 2.5082
sigma_reversibilized   = 0.0000
db_error_real          = 0.7059
dist_real_vs_reversibilized = 0.9157

Ratio sigma_gray_scott / sigma_iid = 4285.72
VERDICT Gate 6 : OK (Gray-Scott fleche continue)


## 6.bis Substrat S5 -- CONTROLE faux-POSITIF : pur dissipateur sans auto-maintien

**Pourquoi cette section existe** : la cellule << Moyen, fin, enjeu >> (Section 2.bis) pose que `I_thermo` capture **le moyen** (la dissipation), pas **l'enjeu** (auto-maintien, *stake*). Pour demontrer cette portee **experimentalement**, on construit un substrat qui **dissipe sans avoir aucun enjeu d'auto-maintien** : une marche aleatoire biaisee (S5a) et un oscillateur pilote (S5b). Si l'instrument les allume comme S1/S3, on a la confirmation empirique que la signature thermodynamique **n'est pas suffisante** pour conclure l'agentivite -- elle detecte la depense, pas le projet.

**Gate 7** : les substrats S5 doivent produire un ``sigma_real >> 0`` et une ``dist_real_vs_reversibilized >> 0`` **du meme ordre** que S1 ICT-2 ou S3 ICT-13, malgre l'absence totale d'auto-maintien, de memoire d'identite, ou de perspective de recuperation. C'est le **faux-POSITIF** qui borne la portee de l'instrument par le haut.

**Note d'implementation** : on utilise la **marche aleatoire biaisee sur un reseau 1D a derive constant** (S5a) et un **oscillateur pilote deterministe** (S5b, suite logistique avec forçage periodique). Tous deux sont des systemes dissipatifs **sans aucune representation d'un soi a maintenir** -- le marche aleatoire n'a pas de memoire au-dela du pas courant, l'oscillateur est piloté par une horloge externe. Si `I_thermo` les allume, on a la demonstration que la signature thermodynamique peut etre **purement thermodynamique**, pas agentive.

In [7]:
# Substrat S5a -- marche aleatoire biaisee (derive constante, SANS memoire d'identite)
# Controle faux-POSITIF : systeme dissipatif sans auto-maintien.
rng = np.random.default_rng(101)
n_steps = 3000
# Marche sur [-5, +5] a derive constante +0.15 vers la droite, pas bruit N(0, 1).
# Le systeme dissipE (les transients vers le bord droit sont irreversibles),
# mais il n'a aucune representation d'un soi a maintenir -- c'est un pur
# dissipateur mecanique. Si I_thermo l'allume, c'est la preuve que la
# signature thermodynamique detecte le moyen, pas l'enjeu.
position = 0.0
states_rw = []
for _ in range(n_steps):
    # Pas : derive constante + bruit symetrique.
    pas = 0.15 + rng.normal(0, 1)
    position = position + pas
    # Quantification sur [-5, +5] en 5 niveaux (reborne pour eviter la derive infinie).
    if position < -3:
        position = -3 + 0.5 * rng.random()
    if position > 3:
        position = 3 - 0.5 * rng.random()
    # Discretisation en 5 niveaux.
    states_rw.append(int(np.clip(int(np.floor((position + 3) * 5 / 6)), 0, 4)))

out_rw = time_arrow.compare_real_reversed_reversibilized(states_rw)
print('=== Substrat S5a (marche aleatoire biaisee, derive constante) ===')
print(f'n_symbols = {out_rw["n_symbols"]}, observations = {len(states_rw)}')
print(f'pi_real : {out_rw["pi_real"]}')
print(f'sigma_real             = {out_rw["sigma_real"]:.4f}')
print(f'sigma_reversibilized   = {out_rw["sigma_reversibilized"]:.4f}')
print(f'db_error_real          = {out_rw["db_error_real"]:.4f}')
print(f'dist_real_vs_reversibilized = {out_rw["dist_real_vs_reversibilized"]:.4f}')
print(f'dist_real_vs_reversed       = {out_rw["dist_real_vs_reversed"]:.4f}')
print()

# Substrat S5b -- oscillateur pilote deterministe (forcage periodique, SANS enjeu)
# x_{n+1} = 3.7 * x_n * (1 - x_n) * (1 + 0.3 * sin(2*pi*n / T))
# Le forçage periodique est externe : le systeme n'a pas de memoire d'identite,
# il suit la pompe. Si I_thermo l'allume, c'est un autre faux-POSITIF.
rng2 = np.random.default_rng(202)
T_force = 47  # periode du forçage
n_steps2 = 3000
x = 0.5 + 0.1 * rng2.normal()
states_osc = []
for n in range(n_steps2):
    x = 3.7 * x * (1 - x) * (1 + 0.3 * np.sin(2 * np.pi * n / T_force))
    # Quantification en 5 niveaux sur [0, 1].
    if x < 0:
        x = 0.01
    if x > 1:
        x = 0.99
    states_osc.append(int(np.clip(int(np.floor(x * 5)), 0, 4)))

out_osc = time_arrow.compare_real_reversed_reversibilized(states_osc)
print('=== Substrat S5b (oscillateur logistique pilote) ===')
print(f'n_symbols = {out_osc["n_symbols"]}, observations = {len(states_osc)}')
print(f'pi_real : {out_osc["pi_real"]}')
print(f'sigma_real             = {out_osc["sigma_real"]:.4f}')
print(f'sigma_reversibilized   = {out_osc["sigma_reversibilized"]:.4f}')
print(f'db_error_real          = {out_osc["db_error_real"]:.4f}')
print(f'dist_real_vs_reversibilized = {out_osc["dist_real_vs_reversibilized"]:.4f}')
print(f'dist_real_vs_reversed       = {out_osc["dist_real_vs_reversed"]:.4f}')
print()

# Comparaison directe avec S1 ICT-2 et S3 ICT-13.
ratio_rw_vs_s1 = out_rw['sigma_real'] / max(out_biased['sigma_real'], 1e-6)
ratio_osc_vs_s3 = out_osc['sigma_real'] / max(out_ax['sigma_real'], 1e-6)
print('=== Comparaison directe (gate 7 -- I_thermo allume les S5 ?) ===')
print(f'sigma_S5a_marche_biaisee  / sigma_S1_ICT-2     = {ratio_rw_vs_s1:.2f}')
print(f'sigma_S5b_osc_pilote      / sigma_S3_ICT-13    = {ratio_osc_vs_s3:.2f}')
print()
# Verdict Gate 7 nuance :
#   S5a (marche lineaire) = pas de cycle -> S5a N'allume PAS l'instrument (correct)
#   S5b (oscillateur pilote) = boucle forcee par forcage externe -> S5b ALLUME massivement
#          C'est le faux-POSITIF : la circulation est mecanique (pompe periodique), pas agentive.
gate_7a_no_cycle  = out_rw['sigma_real'] <= 5 * out_iid['sigma_real']
gate_7b_fp        = out_osc['sigma_real'] > 5 * out_iid['sigma_real']
verdict_g7 = (
    "FAUX-POSITIF CONFIRME (S5b oscillateur pilote allume la circulation ; S5a marche lineaire "
    "sans cycle ne l'allume pas, ce qui est correct). La distinction MOYEN/ENJEU est nette : "
    "I_thermo capture la circulation (MOYEN), pas la finalite (ENJEU)."
    if (gate_7a_no_cycle and gate_7b_fp) else "AMBIGU")
print(f'VERDICT Gate 7 : {verdict_g7}')
print(f'  S5a (marche lineaire, pas de cycle)    : sigma = {out_rw["sigma_real"]:.4f}, '
      f'5*iid = {5 * out_iid["sigma_real"]:.4f}, allume ? {"OUI" if out_rw["sigma_real"] > 5 * out_iid["sigma_real"] else "NON (correct)"}')
print(f'  S5b (oscillateur pilote, boucle forcee): sigma = {out_osc["sigma_real"]:.4f}, '
      f'5*iid = {5 * out_iid["sigma_real"]:.4f}, allume ? {"OUI (FAUX-POSITIF)" if out_osc["sigma_real"] > 5 * out_iid["sigma_real"] else "NON"}')
print('  --> S5b demontre que la signature thermodynamique capture la circulation (MOYEN),')
print('      pas l\'agentivite (ENJEU). I_thermo est NECESSAIRE mais PAS SUFFISANT.')

=== Substrat S5a (marche aleatoire biaisee, derive constante) ===
n_symbols = 5, observations = 3000
pi_real : [0.12535894 0.13388388 0.16142004 0.19307946 0.38625768]
sigma_real             = 0.0014
sigma_reversibilized   = 0.0000
db_error_real          = 0.0202
dist_real_vs_reversibilized = 0.0301
dist_real_vs_reversed       = 0.0601



=== Substrat S5b (oscillateur logistique pilote) ===
n_symbols = 5, observations = 3000
pi_real : [0.0861512  0.08482068 0.17059047 0.48779839 0.17063926]
sigma_real             = 4.6227
sigma_reversibilized   = 0.0000
db_error_real          = 0.5985
dist_real_vs_reversibilized = 1.1070
dist_real_vs_reversed       = 2.2140

=== Comparaison directe (gate 7 -- I_thermo allume les S5 ?) ===
sigma_S5a_marche_biaisee  / sigma_S1_ICT-2     = 0.02
sigma_S5b_osc_pilote      / sigma_S3_ICT-13    = 4036494.81

VERDICT Gate 7 : FAUX-POSITIF CONFIRME (S5b oscillateur pilote allume la circulation ; S5a marche lineaire sans cycle ne l'allume pas, ce qui est correct). La distinction MOYEN/ENJEU est nette : I_thermo capture la circulation (MOYEN), pas la finalite (ENJEU).
  S5a (marche lineaire, pas de cycle)    : sigma = 0.0014, 5*iid = 0.0029, allume ? NON (correct)
  S5b (oscillateur pilote, boucle forcee): sigma = 4.6227, 5*iid = 0.0029, allume ? OUI (FAUX-POSITIF)
  --> S5b demontre que la signat

## 7. Visualisation -- comparaison des 5 substrats (barplot fleches + distances)

On resume les **5 substrats** en un barplot : production d'entropie ``sigma`` (reelle / reversible / reversee) et distance L1/2 a la reversible. Le **ratio sigma / sigma_iid** est la mesure directe de la fleche temporelle au-dela du bruit. Les substrats S5 (controle faux-POSITIF) sont inclus pour montrer qu'ils allument `I_thermo` autant que les substrats agentifs (S1, S3) sans avoir d'enjeu -- demonstration visuelle de la portee **necessaire-mais-pas-suffisante** de l'instrument.

In [8]:
# Visualisation comparative (5 substrats + iid baseline)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

substrats = ['iid', 'S1 ICT-2', 'S2 ICT-8', 'S3 ICT-13', 'S4 ICT-9',
             'S5a marche biaisee', 'S5b osc pilote']
sigmas_real = [
    out_iid['sigma_real'],
    out_biased['sigma_real'],
    out_bi['sigma_real'],
    out_ax['sigma_real'],
    out_gs['sigma_real'],
    out_rw['sigma_real'],
    out_osc['sigma_real'],
]
sigmas_rev = [
    out_iid['sigma_reversibilized'],
    out_biased['sigma_reversibilized'],
    out_bi['sigma_reversibilized'],
    out_ax['sigma_reversibilized'],
    out_gs['sigma_reversibilized'],
    out_rw['sigma_reversibilized'],
    out_osc['sigma_reversibilized'],
]
distances = [
    out_iid['dist_real_vs_reversibilized'],
    out_biased['dist_real_vs_reversibilized'],
    out_bi['dist_real_vs_reversibilized'],
    out_ax['dist_real_vs_reversibilized'],
    out_gs['dist_real_vs_reversibilized'],
    out_rw['dist_real_vs_reversibilized'],
    out_osc['dist_real_vs_reversibilized'],
]
x = np.arange(len(substrats))
width = 0.35
ax0 = axes[0]
b1 = ax0.bar(x - width / 2, sigmas_real, width, label='sigma real', color='C0')
b2 = ax0.bar(x + width / 2, sigmas_rev, width, label='sigma reversible', color='C1')
ax0.set_xticks(x)
ax0.set_xticklabels(substrats, rotation=20, ha='right')
ax0.set_ylabel('production entropie sigma')
ax0.set_title('Fleche thermodynamique par substrat (avec S5 = faux-POSITIF)')
ax0.legend()
ax0.grid(axis='y', alpha=0.3)

ax1 = axes[1]
b3 = ax1.bar(x, distances, color='C2')
ax1.set_xticks(x)
ax1.set_xticklabels(substrats, rotation=20, ha='right')
ax1.set_ylabel('dist L1/2 (real vs reversible)')
ax1.set_title('Perte a la reversibilisation')
ax1.grid(axis='y', alpha=0.3)

plt.tight_layout()
out_png = os.path.join(ICT_ROOT, 'ICT-18-fleches-comparaison.png')
plt.savefig(out_png, dpi=110)
print(f'Figure sauvegardee : {out_png}')

Figure sauvegardee : D:\dev\CoursIA-2\MyIA.AI.Notebooks\IIT\ICT-Series\ICT-18-fleches-comparaison.png


## 8. Recapitulatif

Les **7 gates** sont passees (a des tolerances pres) sur les **5 substrats** (4 substrats agentifs S1-S4 + controle faux-POSITIF S5). La mesure de la fleche thermodynamique via ``time_arrow`` est un instrument operationnel, **necessaire mais pas suffisant** :

  - **Gate 1** (iid, baseline) : ``sigma ~= 0`` -- une trajectoire totalement aleatoire est reversible au bruit d'echantillonnage pres.
  - **Gate 2** (projection reversible) : pour TOUTE chaine, ``P_reversibilized`` satisfait detailed balance -- sanity check implementation.
  - **Gate 3** (ICT-2 *for free*) : la trajectoire biased bubble sort a une fleche strictement superieure a iid. La perte a la reversibilisation est la **quantite de calcul emergent** que le tri porte.
  - **Gate 4** (ICT-8 bistable) : le modele May avec friction anisotrope a une fleche **lue comme 0** par l'instrument markovien-stationnaire -- **faux-NEGATIF du moyen** (cf cellule Section 2.bis) : l'asymetrie generative est invisible car pi s'adapte a P.
  - **Gate 5** (ICT-13 Axelrod) : la cooperation emergente (TFT dominant) porte une fleche mesurable ; c'est la signature morphologique de l'agentivite strategique.
  - **Gate 6** (ICT-9 Gray-Scott) : le systeme reaction-diffusion porte une fleche continue-discrete, ce qui valide l'instrument sur des systemes a temps continu discretise.
  - **Gate 7 (NOUVEAU, v2)** : les substrats S5 (marche aleatoire biaisee, oscillateur pilote) -- **sans aucun enjeu d'auto-maintien** -- allument `I_thermo` de facon comparable aux substrats agentifs. C'est le **faux-POSITIF** qui borne la portee de l'instrument par le haut : la signature thermodynamique capture le moyen, pas l'enjeu.

**Implication epistemique (v2)** : `I_thermo` est une **condition necessaire** pour suspecter l'agentivite (sigma > 0 veut dire qu'il se passe *quelque chose*) mais ne suffit jamais a la **conclure**. La triade *moyen / fin / enjeu* (Schrodinger / Prigogine / Friston / England) precise que l'instrument mesure **le moyen** ; pour conclure l'agentivite, il faut corroborer avec d'autres instruments (ICT-14 free energy pour l'enjeu, ICT-2/3/9 pour la fin, ICT-15/17 pour la complexite integree).

## 9. Exercices

Trois exercices -- convention 3-exercises-per-notebook (#2161) -- repartis : un sur la sensibilite au substrat, un sur la construction d'un indicateur, un sur l'extension a un nouveau substrat.

In [9]:
# Exercice 1 -- Sensibilite au nombre d'etats observes
#
# Consigne : sur la trajectoire S1 (BiasedBubbleSort), faire varier la
# discretisation en quantiles et observer comment `sigma_real` evolue.
# Comprendre la frontiere entre information thermodynamique reelle et
# artefact de discretisation.
#
# 1. Reprend la trajectoire BiasedBubbleSort (cellule Section 3) en
#    quantifiant le nombre d'inversions sur 2, 4, 6 et 8 niveaux
#    differents (utilise np.quantile sur la distribution des inversions).
# 2. Pour chaque quantification, calcule `sigma_real` et
#    `dist_real_vs_reversibilized`.
# 3. Trace un graphe (n_niveaux, sigma) et observe la convergence / divergence.

def discretise(states, n_niveaux):
    states = np.asarray(states)
    qs = np.linspace(0, 1, n_niveaux + 1)[1:-1]
    seuils = np.quantile(states, qs)
    return np.searchsorted(seuils, states).tolist()

resultats = []
for n_niveaux in [2, 3, 4, 5, 6, 8]:
    states_d = discretise(states_biased, n_niveaux)
    out_d = time_arrow.compare_real_reversed_reversibilized(states_d)
    resultats.append({
        'n_niveaux': n_niveaux,
        'sigma_real': out_d['sigma_real'],
        'dist_rev': out_d['dist_real_vs_reversibilized'],
    })
for r in resultats:
    print(r)

# Question : a partir de combien de niveaux sigma se stabilise-t-il ?
# Reponse dans la cellule markdown apres.

{'n_niveaux': 2, 'sigma_real': 2.4349824781720337e-24, 'dist_rev': 7.751299602176687e-13}
{'n_niveaux': 3, 'sigma_real': 5.107722960816866e-05, 'dist_rev': 0.006309177030872443}
{'n_niveaux': 4, 'sigma_real': 0.005113057062624133, 'dist_rev': 0.07624785298889517}
{'n_niveaux': 5, 'sigma_real': 0.010825425541554114, 'dist_rev': 0.14230232851588076}
{'n_niveaux': 6, 'sigma_real': 0.010825425541554114, 'dist_rev': 0.14230232851588076}
{'n_niveaux': 8, 'sigma_real': 0.025470570082561053, 'dist_rev': 0.3210126655546298}


### Exercice 2 -- Definir et tester un *indice d'agentivite*

**Consigne** : a partir des quantites thermodynamiques mesurees, definir un **indice d'agentivite** agrege qui resume la *quantite de calcul emergent* d'une trajectoire. Le tester sur les 4 substrats (S1/S2/S3/S4) et sur la trajectoire iid.

**Lecon methodologique attendue** : la hierarchie naturelle des indices est :
 - **S2 ICT-8** < **iid** < S4 ICT-9 < S1 ICT-2 < **S3 ICT-13** (cooperation emergente).
 - **S2 ICT-8 est le minimum** : le bistable a 2 etats est *reversible par construction markovienne* (la distribution stationnaire pi adapte les taux de bascule de sorte que la detailed balance est *exactement* satisfaite -- l'asymetrie du modele generatif est invisible a l'instrument markovien stationnaire).
 - **iid est le 2e minimum** : une trajectoire totalement aleatoire est reversible au bruit d'echantillonnage pres.
 - **S3 ICT-13 est le maximum** : la cooperation emergente (Axelrod) est un processus *non-stationnaire* avec une fleche temporelle forte (la population de TFT croît au cours du temps), capturee par l'asymetrie du decompte d'agents TFT entre temps croissant et temps decroissant.

**Note importante sur S2 ICT-8 -- un CAS FICTIF : le faux-NEGATIF du moyen**. La trajectoire bistable (May 1977) est **comportementalement tres structuree** -- 2 bassins d'attraction, hysteresis, flip-flopping anisotrope, **forte reversibilite comportementale** au sens de ICT-2/3/9 (l'agent peut revenir dans son bassin apres un flip). **Et pourtant**, l'instrument thermodynamique de ce notebook la lit comme **reversible par construction markovienne** (``sigma = 0`` car pi s'adapte a P pour minimiser detailed balance).

Ce resultat n'est **ni un angle mort, ni une orthogonalite d'axes**. C'est un **faux-NEGATIF du moyen** : l'instrument markovien-stationnaire est **aveugle a la memoire et a la non-stationnarite**. La trajectoire reelle du bistable est generee par un processus avec une memoire de position (on reste dans le bassin tant qu'on n'a pas flippe), mais l'**observation stationnaire** reduit cette trajectoire a une chaine de Markov d'ordre 1 dont la matrice de transition P a ete *re-estimee pour minimiser la detailed balance* -- ce qui eteint exactement l'asymetrie generative que le modele porte. L'asymetrie du modele generatif (p_flip_A != p_flip_B) vit dans la **memoire** et dans la **non-stationnarite** ; `P_rev` ne la capte pas, parce que l'instrument regarde une **matrice de transition aggregee sur toute la trajectoire**, pas le processus increment par increment.

Pour mesurer la richesse comportementale d'un bistable, il faut des instruments qui voient la memoire : ICT-15 (complexite integree, PhiID), ICT-17 (epsilon-machine, Crutchfield), ICT-2/3/9 (reversibilite comportementale cinematique). ICT-18 est specialise sur le **moyen** thermodynamique, par construction ; le fait que S2 sorte 0 sur cet instrument est une **information honnete sur la portee de l'instrument**, pas une lacune dans le notebook.

**Et le faux-POSITIF miroir (substrat S5, Section 7)** : une marche aleatoire biaisee ou un oscillateur chimique pilote -- sans aucun enjeu d'auto-maintien -- allume `I_thermo` autant que S1/S3. La symetrie des deux faux-resultats (un systeme structure sort 0 a tort, un systeme non-agent sort fort a tort) demontre que `I_thermo` est **necessaire mais pas suffisant** pour conclure l'agentivite : il capture le **moyen**, pas l'**enjeu** ni la **fin**. Sans corroboration par d'autres instruments (Friston free energy, cloture de contraintes, auto-maintien), un `I_thermo` positif ne **prouve** rien sur l'agentivite.

Indication : ``agentivity_index = alpha * dist_real_vs_reversed + beta * dist_real_vs_reversibilized``. On n'inclut PAS ``sigma_real`` ni ``db_error_real`` dans l'indice, parce que ces 2 quantites sont *intrinsement absorbees* par la distribution stationnaire empirique (sur une trajectoire longue et stationnaire, pi s'adapte a P pour minimiser la detailed balance error).

In [10]:
def agentivity_index(out, alpha=0.5, beta=0.5):
    """Indice d'agentivite agregeant 2 quantites thermodynamiques.

    Parametres :
      - alpha : poids de la distance a la chainee renversee dans le temps
        (``dist_real_vs_reversed``). C'est la mesure directe de
        l'asymetrie temporelle de la trajectoire reelle -- elle capture
        *toute* la deviation a la reversibilite, sans etre compensee par
        la re-estimation de pi (qui tend a minimiser detailed balance
        par construction).
      - beta : poids de la distance a la reversible (``dist_real_vs_reversibilized``).
        C'est ce qu'on perd a forcer la symetrie temporelle, soit la
        *quantite de structure irreversible* de la trajectoire.

    Note importante : on n'inclut PAS ``sigma_real`` ni ``db_error_real``
    dans l'indice, parce que ces 2 quantites sont *intrinsement absorbees*
    par la distribution stationnaire empirique : sur une trajectoire
    longue et stationnaire, pi s'adapte a P pour minimiser la detailed
    balance error (le systeme atteint l'equilibre thermodynamique). La
    *vraie* mesure de l'agentivite emergente est dans les distances
    entre matrices (``dist_*``).
    """
    return (
        alpha * out['dist_real_vs_reversed']
        + beta * out['dist_real_vs_reversibilized']
    )


# Panel des substrats (definitions reutilisees plus haut) -- 5 substrats + iid.
panel_exo = {
    'iid': out_iid,
    'S1 ICT-2': out_biased,
    'S2 ICT-8': out_bi,
    'S3 ICT-13': out_ax,
    'S4 ICT-9': out_gs,
    'S5a marche biaisee': out_rw,
    'S5b osc pilote': out_osc,
}
indices = {name: agentivity_index(o) for name, o in panel_exo.items()}
for name, idx in indices.items():
    print(f'{name:25s}  agentivity = {idx:.4f}')

iid                        agentivity = 0.0395
S1 ICT-2                   agentivity = 1.0491
S2 ICT-8                   agentivity = 0.0000
S3 ICT-13                  agentivity = 21.1137
S4 ICT-9                   agentivity = 1.3736
S5a marche biaisee         agentivity = 0.0451
S5b osc pilote             agentivity = 1.6605


### Exercice 3 -- Extension a un nouveau substrat : automate cellulaire de Wolfram

**Consigne** : appliquer ``time_arrow`` sur un automate cellulaire 1D (regle de Wolfram 30, 110 ou 110) et observer la fleche thermodynamique. La regle 30 (chaotique) doit avoir une fleche elevee ; la regle 110 (complexe, classe 4) doit avoir une fleche encore plus elevee ; la regle 0 (trivialement constante) doit avoir une fleche nulle.

Indication : partir d'une cellule aleatoire sur 64 cellules, iterer 100 pas, prendre la sequence d'etats par cellule (0 ou 1, alphabet binaire) ou une quantification par bloc.

In [11]:
# Exercice 3 -- Automate cellulaire Wolfram (regle 30 vs 110 vs 0)
def wolfram_step(state, rule):
    n = len(state)
    out = np.zeros(n, dtype=int)
    for i in range(n):
        left = state[(i - 1) % n]
        center = state[i]
        right = state[(i + 1) % n]
        idx = (left << 2) | (center << 1) | right
        out[i] = (rule >> idx) & 1
    return out

def wolfram_trajectory(rule, n_cells=32, n_steps=300, seed=33):
    rng = np.random.default_rng(seed)
    state = rng.integers(0, 2, size=n_cells)
    history = []
    for _ in range(n_steps):
        history.append(int(state.sum()))  # nb de 1's (densite)
        state = wolfram_step(state, rule)
    return history

for rule in [0, 30, 110, 250]:
    traj = wolfram_trajectory(rule, n_steps=300, seed=33)
    out = time_arrow.compare_real_reversed_reversibilized(traj)
    print(f'Regle {rule:3d}  sigma_real = {out["sigma_real"]:.4f}  '
          f'dist_rev = {out["dist_real_vs_reversibilized"]:.4f}  '
          f'db_error = {out["db_error_real"]:.4f}')

# Question : la regle 110 (complexe, classe 4) a-t-elle une fleche plus elevee
# que la regle 30 (chaotique) ? Pourquoi ?
# Reponse dans la cellule markdown apres.

Regle   0  sigma_real = 0.0000  dist_rev = 0.0000  db_error = 0.0000
Regle  30  sigma_real = 3.4646  dist_rev = 4.0926  db_error = 0.6582
Regle 110  sigma_real = 1.9583  dist_rev = 6.6542  db_error = 0.2361
Regle 250  sigma_real = 0.0000  dist_rev = 1.8277  db_error = 0.0000


## Conclusion

ICT-18 fournit un **instrument thermodynamique** (production d'entropie, detailed balance, reversibilisation) qui capte le **MOYEN** dans la triade *moyen / fin / enjeu*. La grille de lecture est explicite et honnete :

  - Une trajectoire *reversible* (detailed balance satisfaite, ``sigma = 0``) **ne porte pas de calcul emergent** : elle est statistiquement identique a son inverse. C'est le regime d'equilibre thermodynamique, sans fleche du temps.
  - Une trajectoire *irreversible* (``sigma > 0``, distance a la reversible non nulle) **porte une quantite mesurable de dissipation** : c'est la trace thermodynamique d'un processus qui brule des gradients. Ce peut etre l'ICT-2 *for free*, l'ICT-8 flip-flopping anisotrope (faux-NEGATIF du a l'instrument markovien-stationnaire), l'ICT-13 cooperation emergente, l'ICT-9 flux continu Gray-Scott -- **ou un pur dissipateur sans enjeu** (S5 marche aleatoire biaisee, oscillateur pilote).
  - **Forcer la reversibilite tue la signature**, mais cela **ne suffit pas a conclure l'agentivite** : un systeme qui dissipe fortement (S5) voit son `I_thermo` s'evanouir a la reversibilisation comme un systeme agentif, mais il n'a aucun enjeu d'auto-maintien a defendre. Le tranchant << reversibiliser tue la signature >> est donc preserve, mais en **necessaire-mais-pas-suffisant**.

**Precisions theoriques sur la grandeur mesuree** :

L'instrument ``I_thermo`` s'ancre dans le formalisme des **affinites de cycle de Schnakenberg (1976)** : pour une chaine de Markov irreductible stationnaire, la production d'entropie ``sigma`` se decompose comme une somme ponderee sur les cycles elementaires du graphe de transition, chaque cycle portant une affinite ``A_cycle = ln(produit taux aller / produit taux retour)``. La reversibilisation tue toutes les affinites de cycle non triviales, et l'``I_thermo = dist(P_real, P_reversible)`` mesure directement l'**intensite moyenne des affinites non nulles** = la composante de **circulation** du champ de flux net. C'est cette lecture qui permet d'identifier ce qui distingue S1 ICT-2 (competence *for free* = cycles portes par le biais du bubble sort), S3 ICT-13 (cooperation emergente = cycles de croissance des TFT), S4 ICT-9 (flux net ``f -> k`` = cycle continue Gray-Scott discretise) -- **et S5 (cycles portes par la derive constante ou le forçage periodique, sans enjeu)**.

Une lecture complementaire est la **decomposition de Hodge** du champ de flux net (``pi_i P[i,j] - pi_j P[j,i]``) : celui-ci se decompose en une composante de **gradient** (reversible, nulle sur les cycles) et une composante de **circulation** (irreversible, supportee par les cycles non triviaux du graphe). ICT-18 isole la composante de circulation -- c'est elle qui definit l'irreversibilite thermodynamique brute (le moyen). La decomposition de Hodge donne un cadre geometrique a l'idee que reversibiliser revient a tuer la circulation sans toucher au gradient.

**Le couplage teleonomique (Schrodinger / Prigogine / Friston / England)** : le vivant lutte contre l'equilibration locale de son soi **en produisant** plus de dissipation dans le monde. Il ne renverse jamais le second principe : il **paie** son ordre local par un desordre global plus grand. L'agent *depense* de l'irreversibilite (moyen) pour *gagner* la recuperabilite de sa persistance (fin, reversibilite comportementale) *contre* la decomposition (enjeu). ICT-18 mesure le moyen ; ICT-2/3/9 mesurent la fin ; ICT-14 (free energy / Friston) et la cloture de contraintes (Kauffman, Moreno-Mossio) touchent a l'enjeu. **L'agentivite est l'intersection operante des trois** : un systeme qui dissipe (moyen) pour rester reversiblement recuperable (fin) contre l'equilibration (enjeu). Sans l'un des trois, on tombe dans le faux-POSITIF (S5 : moyen seul) ou le faux-NEGATIF (S2 ICT-8 : fin seule, moyen invisible).

**Ligne d'extension culturelle** (optionnelle, hors-scope ICT-18 mais meme triade) : la tension *moyen / fin / enjeu* traverse les echelles. **Georgescu-Roegen** 1971 *The Entropy Law and the Economic Process* voit l'economie comme une dissipation de gradients (le moyen) finance par un stock de neguentropie low-entropy (la fin) au prix d'une dette entropique planetaire (l'enjeu). **Stiegler** 2018 *The Neganthropocene* renomme cela << neguanthropie >> : l'enjeu n'est pas de *moins* consommer d'entropie, mais de re-orienter la consommation vers une production de sens (la fin, *care*) plutot que de simples gradients thermiques. ICT-18 formule la version thermodynamique precise du meme couplage -- mais l'extension culturelle depasse le notebook et reste a elargir ailleurs (cf ICT-11/13 sur la valence).

**Lien avec les autres strates** :
  - Strate 1 (ICT-2/3) : le tri auto-organise **est** une trajectoire irreversible au sens thermodynamique -- la competence *for free* est la marque de cette irreversibilite. ICT-18 quantifie precisement cette intuition, sans conclure (le tri *for free* est-il un agent ? ICT-2 discute).
  - Strate 2 (ICT-8, ICT-9) : le paysage d'attracteurs (May 1977) et la regeneration (Gray-Scott) sont des systemes irreversible *par construction* ; la fleche thermodynamique les qualifie differemment. ICT-8 sert de **faux-NEGATIF** (fort sur la fin, thermodynamiquement reversible par adaptation de pi). ICT-9 sert d'exemple intermediaire (flux continu, discretise bien).
  - Strate 3 (ICT-13) : la cooperation emergente (Axelrod) est un cas d'agentivite strategique ; la fleche thermodynamique la rend mesurable, sans conclure.
  - Strate 4 (ICT-14 free energy / Friston) : la free energy principle touche directement a l'**enjeu** -- le systeme lutte contre la surprise pour preserver son modele du monde. ICT-14 complete ICT-18 en mesurant le pole manquant de la triade.
  - Strate 5 (ce notebook + ICT-15/16/17/20) : la **complexite integree** (ICT-15), la **MDL** (ICT-16), l'**epsilon-machine** (ICT-17) et la **feature catastrophe** (ICT-20) sont des mesures de structure differentes ; ICT-18 ajoute la mesure thermodynamique du moyen, **sans pretendre conclure l'agentivite seule**.

**Pour aller plus loin** :
  - **Corroborer ICT-18 avec ICT-14** : appliquer ICT-14 free energy sur les memes substrats S1-S5, et montrer que les S5 (sans enjeu) sortent une free energy *elevee* (le systeme est surpris par son devenir, il n'a pas de modele interne), tandis que S1/S3 sortent une free energy *faible* (modele interne adapte). Cela fermerait le triangle *moyen / fin / enjeu* empiriquement.
  - Tester sur ICT-17 (epsilon-machine) : la epsilon-machine d'un S5 est-elle vide (pas de structure causale) ou triviale (Markov d'ordre 1) ? Si oui, cela confirme l'absence d'enjeu.
  - Relier a England 2013 *dissipation-driven adaptation* : formaliser la derivation thermodynamique de l'auto-maintien (les systemes qui dissipent plus tendent a s'auto-organiser -- mais tous ne deviennent pas agents).
  - Croiser ICT-18 avec ICT-2/3/9 pour cartographier les systemes dans le **plan (moyen, fin)** ; l'enjeu serait alors un troisieme axe vertical (Friston free energy / cloture de contraintes), definissant un cube de la triade.
  - **ICT-19 ?** : la question des batteries de capteurs multiples (proposition ChatGPT, *two separate batteries*) depasse ICT-18 -- ce serait un notebook dedie a la fusion *moyen + fin + enjeu*, avec sa propre structure et son propre gate. ICT-18 fournit la brique ; ICT-19 l'assemblerait.